# Model Evaluation & Benchmark Comparison: Old Model (`model.plan`) vs New Model (RF-DETR)

This dedicated evaluation pipeline compares your **existing TensorRT engine** (`model.plan`) against your **new fine-tuned RF-DETR model** (`.pth` checkpoint).

### Key Capabilities:
1. **Universal TensorRT Engine Support**: Native zero-copy TensorRT runtime via PyTorch CUDA tensors (with optional Ultralytics YOLO fallback).
2. **Rigorous Accuracy Evaluation**: COCO mAP (`mAP@50:95`, `mAP@50`, `mAP@75`, `mAP_s`, `mAP_m`, `mAP_l`, and per-class AP) using `torchmetrics.detection.mean_ap.MeanAveragePrecision`.
3. **Speed & Latency Profiling**: GPU CUDA Event benchmarking measuring Mean Latency (ms), P95 Latency, and Throughput (FPS).
4. **Side-by-Side Metric Delta Table**: Automated comparison table showing exact performance gains/deltas (+/- difference and % change).
5. **3-Panel Visual Comparison & Disagreement Analysis**: Side-by-side visualization of `[Ground Truth]` vs `[Old Model (model.plan)]` vs `[New Model (RF-DETR)]` to pinpoint edge-case errors.

In [ ]:
# STEP 0 - Verify and Install Dependencies
print("=" * 70)
print("[STEP 0] Checking & installing required evaluation dependencies...")
print("=" * 70)

!pip install -q --no-cache-dir torchmetrics supervision pycocotools pandas tabulate matplotlib opencv-python-headless
try:
    import tensorrt
    print(f"TensorRT is available: version {tensorrt.__version__}")
except ImportError:
    print("Notice: tensorrt Python module not found. Installing nvidia-tensorrt...")
    !pip install -q tensorrt

print("Dependencies check completed.")


In [ ]:
# CELL 1 - Imports, Logger Setup & CUDA VRAM Diagnostics
import os, sys, json, time, math, copy, logging, random, shutil, warnings
warnings.filterwarnings("ignore", category=FutureWarning)
from pathlib import Path
from typing import Dict, List, Any, Optional, Tuple, Union
from collections import defaultdict

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(line_buffering=True)
os.environ["PYTHONUNBUFFERED"] = "1"

# Prevent file-descriptor starvation in multi-worker loaders
import torch.multiprocessing as mp
try:
    mp.set_sharing_strategy('file_system')
except Exception:
    pass

try:
    import resource
    rlimit = resource.getrlimit(resource.RLIMIT_NOFILE)
    resource.setrlimit(resource.RLIMIT_NOFILE, (max(rlimit[0], 4096), max(rlimit[1], 4096)))
except Exception:
    pass

import cv2, torch, numpy as np, pandas as pd
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
from PIL import Image, ImageDraw, ImageFont
from tqdm.auto import tqdm
from IPython.display import display, Image as IPImage
import supervision as sv
from torchmetrics.detection.mean_ap import MeanAveragePrecision

# Optional TensorRT import
try:
    import tensorrt as trt
    HAS_TRT = True
except ImportError:
    HAS_TRT = False
    trt = None

# Auto-flushing logger
class FlushHandler(logging.StreamHandler):
    def emit(self, record):
        super().emit(record)
        self.flush()

logger = logging.getLogger("compare_evaluations")
logger.setLevel(logging.INFO)
logger.handlers.clear()
logger.propagate = False
console_handler = FlushHandler(sys.stdout)
console_handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)-8s | %(message)s"))
logger.addHandler(console_handler)

def print_vram_usage(tag="Status"):
    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated() / (1024**3)
        total = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        logger.info(f"[VRAM - {tag}] {torch.cuda.get_device_name(0)}: {alloc:.2f} GB / {total:.2f} GB allocated")
    else:
        logger.info(f"[VRAM - {tag}] Running on CPU")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info(f"Running on Device: {device} | TensorRT Available: {HAS_TRT}")
print_vram_usage("Init")


In [ ]:
# CELL 2 - Configuration & Path Setup
logger.info("=" * 70)
logger.info("[CELL 2] Configuring evaluation comparison paths and parameters...")
logger.info("=" * 70)

# 1. Model Paths
PLAN_MODEL_PATH = "model.plan"          # Path to your existing TensorRT engine
CONFIG_PBTXT_PATH = "config.pbtxt"      # Path to Triton config.pbtxt (if available)

# Auto-detect trained RF-DETR model checkpoint
rfdetr_candidates = [
    "location_tag_single_class_train_rfdetr/model/best_model_full_data.pth",
    "location_tag_single_class_train_rfdetr/model/best_model_sample_1000.pth",
    "multi_class_train_rfdetr/model/best_model_full_data.pth",
    "multi_class_train_rfdetr/model/best_model_sample_1000.pth",
    "location_tag_single_class_train_rfdetr/runs/rf_detr_full_data/checkpoints/best_loss.pth",
    "multi_class_train_rfdetr/runs/rf_detr_full_data/checkpoints/best_loss.pth",
]
RFDETR_CHECKPOINT_PATH = next((p for p in rfdetr_candidates if os.path.exists(p)), "best_model_rfdetr.pth")

# 2. Dataset Paths
ann_candidates = [
    ("test_data/test/_annotations.coco.json", "test_data/images"),
    ("location_tag_single_class_train_rfdetr/dataset_full_data/test/_annotations.coco.json", "location_tag_single_class_train_rfdetr/images"),
    ("location_tag_single_class_train_rfdetr/dataset_sample_1000/test/_annotations.coco.json", "location_tag_single_class_train_rfdetr/images"),
    ("multi_class_train_rfdetr/dataset_full_data/test/_annotations.coco.json", "multi_class_train_rfdetr/images"),
    ("multi_class_train_rfdetr/dataset_sample_1000/test/_annotations.coco.json", "multi_class_train_rfdetr/images"),
]
default_ann, default_img_dir = next(((a, img) for a, img in ann_candidates if os.path.exists(a)), ("test_data/test/_annotations.coco.json", "test_data/images"))

EVAL_ANN_PATH = default_ann
EVAL_IMAGES_DIR = default_img_dir

# 3. Evaluation Hyperparameters
RESOLUTION_RFDETR = 560         # Input resolution for RF-DETR (560 for base, or 1008 for high-res)
RESOLUTION_PLAN = None          # None = auto-detect from config.pbtxt or model.plan, or specify int
CONFIDENCE_THRESHOLD = 0.30     # Minimum confidence to accept detection
IOU_THRESHOLD = 0.50            # NMS / mAP IoU overlap threshold
MAX_EVAL_SAMPLES = None         # Set to integer (e.g. 200) for fast testing, or None for full test split
BENCHMARK_ITERS = 50            # Iterations for speed/latency profiling
NUM_VISUAL_PREVIEWS = 10        # Number of side-by-side comparison images to export

# Auto-parse Triton config.pbtxt if present to extract input/output metadata
def parse_triton_pbtxt(pbtxt_path: str) -> dict:
    if not os.path.exists(pbtxt_path):
        return {}
    import re
    info = {'inputs': [], 'outputs': []}
    try:
        with open(pbtxt_path, 'r', encoding='utf-8') as f:
            content = f.read()
        in_matches = re.findall(r'name:\s*"([^"]+)".*?dims:\s*\[\s*([\d\s,]+)\s*\]', content, re.DOTALL)
        # Separate inputs and outputs blocks
        in_part = re.findall(r'input\s*\[(.*?)\]', content, re.DOTALL)
        for block in in_part:
            names = re.findall(r'name:\s*"([^"]+)"', block)
            dims = re.findall(r'dims:\s*\[\s*([\d\s,]+)\s*\]', block)
            for n, d in zip(names, dims):
                dim_list = [int(x.strip()) for x in d.split(',') if x.strip()]
                info['inputs'].append({'name': n, 'dims': dim_list})
        out_part = re.findall(r'output\s*\[(.*?)\]', content, re.DOTALL)
        for block in out_part:
            names = re.findall(r'name:\s*"([^"]+)"', block)
            dims = re.findall(r'dims:\s*\[\s*([\d\s,]+)\s*\]', block)
            for n, d in zip(names, dims):
                dim_list = [int(x.strip()) for x in d.split(',') if x.strip()]
                info['outputs'].append({'name': n, 'dims': dim_list})
    except Exception as err:
        logger.warning(f"Error parsing {pbtxt_path}: {err}")
    return info

triton_metadata = parse_triton_pbtxt(CONFIG_PBTXT_PATH)
if triton_metadata.get('inputs'):
    logger.info(f"Parsed Triton config.pbtxt successfully: {triton_metadata}")
    first_dims = triton_metadata['inputs'][0]['dims']
    if len(first_dims) >= 2:
        RESOLUTION_PLAN = first_dims[-2] if first_dims[-2] > 0 else first_dims[-1]
        logger.info(f"   - Inferred model.plan Resolution from config.pbtxt: {RESOLUTION_PLAN}x{RESOLUTION_PLAN}")

# 4. Output Directory for Results and Visualizations
OUTPUT_DIR = Path("evaluation_comparison")
PREVIEWS_DIR = OUTPUT_DIR / "previews"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PREVIEWS_DIR.mkdir(parents=True, exist_ok=True)

logger.info(f"Configuration:")
logger.info(f"   - Old Model (TRT Plan):   {PLAN_MODEL_PATH} (Exists: {os.path.exists(PLAN_MODEL_PATH)})")
logger.info(f"   - Triton Config (pbtxt):  {CONFIG_PBTXT_PATH} (Exists: {os.path.exists(CONFIG_PBTXT_PATH)})")
logger.info(f"   - New Model (RF-DETR):    {RFDETR_CHECKPOINT_PATH} (Exists: {os.path.exists(RFDETR_CHECKPOINT_PATH)})")
logger.info(f"   - Evaluation COCO JSON:   {EVAL_ANN_PATH} (Exists: {os.path.exists(EVAL_ANN_PATH)})")
logger.info(f"   - Evaluation Images Dir:  {EVAL_IMAGES_DIR} (Exists: {os.path.exists(EVAL_IMAGES_DIR)})")
logger.info(f"   - Confidence Threshold:   {CONFIDENCE_THRESHOLD}")
logger.info(f"   - Output Directory:       {OUTPUT_DIR}")


In [ ]:
# CELL 3 - Load Dataset Annotations and Ground Truth Information
logger.info("=" * 70)
logger.info(f"[CELL 3] Inspecting Evaluation Dataset from: {EVAL_ANN_PATH}")
logger.info("=" * 70)

assert os.path.exists(EVAL_ANN_PATH), f"Annotation file not found at: {EVAL_ANN_PATH}. Please update EVAL_ANN_PATH in Cell 2."

with open(EVAL_ANN_PATH, "r", encoding="utf-8") as f:
    coco_data = json.load(f)

# Build category lookups (0-indexed)
raw_categories = sorted(coco_data.get("categories", []), key=lambda c: c["id"])
cat_id_to_idx = {c["id"]: i for i, c in enumerate(raw_categories)}
idx_to_name = {i: c["name"] for i, c in enumerate(raw_categories)}
class_names = [idx_to_name[i] for i in range(len(raw_categories))]
NUM_CLASSES = len(class_names)

total_images = len(coco_data.get("images", []))
total_annotations = len(coco_data.get("annotations", []))
cat_box_counts = defaultdict(int)
for ann in coco_data.get("annotations", []):
    cat_idx = cat_id_to_idx.get(ann["category_id"], 0)
    cat_box_counts[cat_idx] += 1

logger.info(f"Dataset Summary:")
logger.info(f"   - Total Images:      {total_images}")
logger.info(f"   - Total Annotations: {total_annotations}")
logger.info(f"   - Number of Classes: {NUM_CLASSES} ({class_names})")
for idx, name in enumerate(class_names):
    logger.info(f"     [{idx}] {name:20s}: {cat_box_counts[idx]} ground truth boxes")

# Dataset class for evaluation
class COCOEvalDataset(Dataset):
    def __init__(self, img_dir: str, ann_file: str, resolution: int, max_samples: Optional[int] = None):
        with open(ann_file, "r", encoding="utf-8") as f:
            data = json.load(f)
        self.img_dir = img_dir
        self.resolution = resolution
        self.cat_id_to_idx = {c["id"]: i for i, c in enumerate(sorted(data["categories"], key=lambda x: x["id"]))}
        
        ann_by_img = defaultdict(list)
        for ann in data.get("annotations", []):
            ann_by_img[ann["image_id"]].append(ann)
        
        samples = [(img, ann_by_img[img["id"]]) for img in data["images"] if img["id"] in ann_by_img]
        if max_samples:
            samples = samples[:max_samples]
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_meta, anns = self.samples[idx]
        img_path = os.path.join(self.img_dir, img_meta["file_name"])
        
        # Read original image
        cv_img = cv2.imread(img_path)
        if cv_img is not None:
            orig_h, orig_w = cv_img.shape[:2]
            resized = cv2.resize(cv_img, (self.resolution, self.resolution), interpolation=cv2.INTER_LINEAR)
            rgb = cv2.cvtColor(resized, cv2.COLOR_BGR2RGB)
            img_t = torch.from_numpy(rgb).permute(2, 0, 1).float().div_(255.0)
        else:
            with Image.open(img_path).convert("RGB") as pil_im:
                orig_w, orig_h = pil_im.size
                resized = pil_im.resize((self.resolution, self.resolution), Image.BILINEAR)
                img_t = TF.to_tensor(resized)
        
        # Normalize for PyTorch backbones
        norm_t = TF.normalize(img_t, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        
        boxes_norm, boxes_xyxy_orig, labels = [], [], []
        for a in anns:
            x, y, w, h = a["bbox"]
            if w <= 0 or h <= 0: continue
            boxes_norm.append([
                float(np.clip((x + w / 2) / orig_w, 0, 1)),
                float(np.clip((y + h / 2) / orig_h, 0, 1)),
                float(np.clip(w / orig_w, 0, 1)),
                float(np.clip(h / orig_h, 0, 1))
            ])
            boxes_xyxy_orig.append([x, y, x + w, y + h])
            labels.append(self.cat_id_to_idx[a["category_id"]])
        
        return {
            "norm_tensor": norm_t,
            "raw_rgb_tensor": img_t,  # [0, 1] range for models requiring simple float32 scaling
            "file_name": img_meta["file_name"],
            "image_id": img_meta["id"],
            "orig_size": (orig_h, orig_w),
            "boxes_cxcywh_norm": torch.tensor(boxes_norm, dtype=torch.float32) if boxes_norm else torch.zeros((0, 4)),
            "boxes_xyxy_orig": torch.tensor(boxes_xyxy_orig, dtype=torch.float32) if boxes_xyxy_orig else torch.zeros((0, 4)),
            "labels": torch.tensor(labels, dtype=torch.long) if labels else torch.zeros(0, dtype=torch.long),
        }

eval_dataset = COCOEvalDataset(EVAL_IMAGES_DIR, EVAL_ANN_PATH, resolution=RESOLUTION_RFDETR, max_samples=MAX_EVAL_SAMPLES)
logger.info(f"COCOEvalDataset successfully initialized with {len(eval_dataset)} valid evaluation samples.")


In [ ]:
# CELL 4 - Load Fine-Tuned RF-DETR Model
logger.info("=" * 70)
logger.info(f"[CELL 4] Loading RF-DETR Model Checkpoint: {RFDETR_CHECKPOINT_PATH}")
logger.info("=" * 70)

rfdetr_model = None
if os.path.exists(RFDETR_CHECKPOINT_PATH):
    try:
        from rfdetr import RFDETRBase
        from rfdetr.models.lwdetr import LWDETR
        
        # Safe loader patch
        def _safe_lwdetr_load(self, state_dict, strict=True):
            model_state = self.state_dict()
            filtered = {}
            for k, v in state_dict.items():
                clean_k = k[6:] if k.startswith("model.") else (k[7:] if k.startswith("module.") else k)
                if clean_k in model_state and model_state[clean_k].shape == v.shape:
                    filtered[clean_k] = v
                elif k in model_state and model_state[k].shape == v.shape:
                    filtered[k] = v
            return torch.nn.Module.load_state_dict(self, filtered, strict=False)
        
        LWDETR.load_state_dict = _safe_lwdetr_load
        
        # Build RF-DETR base wrapper
        rf_wrapper = RFDETRBase(num_classes=NUM_CLASSES, resolution=RESOLUTION_RFDETR, pretrained=False)
        rfdetr_model = rf_wrapper.model
        
        # Load weights
        checkpoint = torch.load(RFDETR_CHECKPOINT_PATH, map_location="cpu", weights_only=False)
        state = checkpoint.get("model", checkpoint)
        rfdetr_model.load_state_dict(state, strict=False)
        rfdetr_model.to(device)
        rfdetr_model.eval()
        logger.info(f"RF-DETR Model loaded successfully onto {device} ({sum(p.numel() for p in rfdetr_model.parameters()) / 1e6:.1f}M params).")
    except Exception as e:
        logger.error(f"Failed to load RF-DETR model: {e}")
        rfdetr_model = None
else:
    logger.warning(f"RF-DETR Checkpoint not found at: {RFDETR_CHECKPOINT_PATH}. Please train or specify correct path.")


In [ ]:
# CELL 5 - Universal TensorRT Runner (model.plan)
logger.info("=" * 70)
logger.info(f"[CELL 5] Initializing TensorRT Engine from: {PLAN_MODEL_PATH}")
logger.info("=" * 70)

class TensorRTRunner:
    """
    Universal TensorRT execution wrapper using PyTorch CUDA memory buffers.
    Zero-copy, high performance, and handles dynamic or fixed bindings.
    """
    def __init__(self, plan_path: str, device: torch.device):
        self.plan_path = plan_path
        self.device = device
        self.is_ready = False
        
        if not HAS_TRT:
            logger.warning("TensorRT library is not installed in the current environment.")
            return
            
        if not os.path.exists(plan_path):
            logger.warning(f"TensorRT engine file '{plan_path}' does not exist yet. Please provide the file.")
            return
            
        logger.info(f"Deserializing TensorRT CUDA engine from {plan_path}...")
        trt_logger = trt.Logger(trt.Logger.WARNING)
        with open(plan_path, "rb") as f, trt.Runtime(trt_logger) as runtime:
            self.engine = runtime.deserialize_cuda_engine(f.read())
            
        if self.engine is None:
            logger.error("Failed to deserialize TensorRT engine.")
            return
            
        self.context = self.engine.create_execution_context()
        self.inputs = []
        self.outputs = []
        self.bindings = {}
        self._inspect_io()
        self.is_ready = True
        logger.info(f"TensorRT Engine successfully initialized and ready for inference.")

    def _inspect_io(self):
        """Inspect input and output tensors across TensorRT 8.x and 10.x APIs."""
        # Check if modern TensorRT 8.5+ tensor API is available
        if hasattr(self.engine, "num_io_tensors"):
            num_tensors = self.engine.num_io_tensors
            for i in range(num_tensors):
                name = self.engine.get_tensor_name(i)
                mode = self.engine.get_tensor_mode(name)
                shape = list(self.engine.get_tensor_shape(name))
                dtype = self.engine.get_tensor_dtype(name)
                is_input = (mode == trt.TensorIOMode.INPUT)
                meta = {"name": name, "shape": shape, "dtype": dtype, "is_input": is_input}
                if is_input:
                    self.inputs.append(meta)
                else:
                    self.outputs.append(meta)
                logger.info(f"   - {'[INPUT]' if is_input else '[OUTPUT]'} Tensor '{name}': Shape={shape}, Dtype={dtype}")
        else:
            # Legacy TensorRT bindings API
            for i in range(self.engine.num_bindings):
                name = self.engine.get_binding_name(i)
                is_input = self.engine.binding_is_input(i)
                shape = list(self.engine.get_binding_shape(i))
                dtype = self.engine.get_binding_dtype(i)
                meta = {"name": name, "shape": shape, "dtype": dtype, "is_input": is_input}
                if is_input:
                    self.inputs.append(meta)
                else:
                    self.outputs.append(meta)
                logger.info(f"   - {'[INPUT]' if is_input else '[OUTPUT]'} Binding '{name}': Shape={shape}, Dtype={dtype}")

    def get_input_resolution(self) -> int:
        """Infers input image resolution from the first input tensor shape."""
        if self.inputs and len(self.inputs[0]["shape"]) == 4:
            h = self.inputs[0]["shape"][2]
            return h if h > 0 else 640
        return 640

    def infer(self, input_tensor: torch.Tensor) -> List[torch.Tensor]:
        """
        Runs inference synchronously on GPU with zero memory copy.
        """
        assert self.is_ready, "TensorRT engine is not ready."
        input_tensor = input_tensor.contiguous().to(self.device)
        output_tensors = []
        
        if hasattr(self.context, "set_tensor_address"):
            # Modern TRT 8.5+ API
            self.context.set_tensor_address(self.inputs[0]["name"], input_tensor.data_ptr())
            for out_meta in self.outputs:
                # Resolve dynamic batch size or dims if negative
                shape = [input_tensor.shape[0] if s < 0 and idx == 0 else (abs(s) if s < 0 else s)
                         for idx, s in enumerate(out_meta["shape"])]
                out_t = torch.empty(shape, dtype=torch.float32, device=self.device)
                self.context.set_tensor_address(out_meta["name"], out_t.data_ptr())
                output_tensors.append(out_t)
            
            stream = torch.cuda.current_stream().cuda_stream
            self.context.execute_async_v3(stream)
            torch.cuda.synchronize()
        else:
            # Legacy TRT API
            bindings = [input_tensor.data_ptr()]
            for out_meta in self.outputs:
                shape = [input_tensor.shape[0] if s < 0 and idx == 0 else (abs(s) if s < 0 else s)
                         for idx, s in enumerate(out_meta["shape"])]
                out_t = torch.empty(shape, dtype=torch.float32, device=self.device)
                bindings.append(out_t.data_ptr())
                output_tensors.append(out_t)
            stream = torch.cuda.current_stream().cuda_stream
            self.context.execute_async_v2(bindings=bindings, stream_handle=stream)
            torch.cuda.synchronize()
            
        return output_tensors

trt_runner = TensorRTRunner(PLAN_MODEL_PATH, device)
if trt_runner.is_ready and RESOLUTION_PLAN is None:
    RESOLUTION_PLAN = trt_runner.get_input_resolution()
    logger.info(f"Auto-detected model.plan input resolution: {RESOLUTION_PLAN}x{RESOLUTION_PLAN}")
elif RESOLUTION_PLAN is None:
    RESOLUTION_PLAN = 640


In [ ]:
# CELL 6 - Standardized Prediction Decoders (RF-DETR vs TensorRT Engine)
logger.info("=" * 70)
logger.info("[CELL 6] Compiling prediction decoders...")
logger.info("=" * 70)

def box_cxcywh_to_xyxy(boxes: torch.Tensor) -> torch.Tensor:
    cx, cy, w, h = boxes.unbind(-1)
    return torch.stack([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2], dim=-1)

def decode_rfdetr_outputs(outputs: Dict[str, torch.Tensor], orig_size: Tuple[int, int], conf_thresh: float = 0.3) -> Dict[str, torch.Tensor]:
    """
    Converts raw RF-DETR outputs to original image coordinate space.
    """
    orig_h, orig_w = orig_size
    pred_logits = outputs["pred_logits"][0]  # [queries, num_classes]
    pred_boxes = outputs["pred_boxes"][0]    # [queries, 4] normalized cxcywh
    
    probs = pred_logits.sigmoid()
    scores, labels = probs.max(dim=-1)
    keep = scores > conf_thresh
    
    if keep.sum() == 0:
        return {
            "boxes": torch.zeros((0, 4), dtype=torch.float32),
            "scores": torch.zeros((0,), dtype=torch.float32),
            "labels": torch.zeros((0,), dtype=torch.long),
        }
        
    boxes_xyxy_norm = box_cxcywh_to_xyxy(pred_boxes[keep])
    scale = torch.tensor([orig_w, orig_h, orig_w, orig_h], device=boxes_xyxy_norm.device)
    boxes_xyxy_orig = boxes_xyxy_norm * scale
    
    return {
        "boxes": boxes_xyxy_orig.cpu().float(),
        "scores": scores[keep].cpu().float(),
        "labels": labels[keep].cpu().long(),
    }

def decode_plan_outputs(raw_outputs: List[torch.Tensor], orig_size: Tuple[int, int], plan_resolution: int, conf_thresh: float = 0.3, iou_thresh: float = 0.5) -> Dict[str, torch.Tensor]:
    """
    Universal decoder for model.plan output tensors:
    Supports YOLOv8/v5 [1, 4+C, N], YOLOv8 transposed [1, N, 4+C], DETR-like [1, N, 4] & [1, N, C],
    and NMS-integrated engines [1, N, 6].
    """
    orig_h, orig_w = orig_size
    
    if not raw_outputs:
        return {"boxes": torch.zeros((0, 4)), "scores": torch.zeros((0,)), "labels": torch.zeros((0,), dtype=torch.long)}
    
    out = raw_outputs[0]
    # Case 1: YOLOv8 shape [1, 4 + num_classes, num_anchors] -> Transpose to [1, num_anchors, 4 + num_classes]
    if out.ndim == 3 and out.shape[1] < out.shape[2] and out.shape[1] <= 100:
        out = out.permute(0, 2, 1)
        
    # Case 2: Standard detection output [1, num_boxes, 4 + num_classes]
    if out.ndim == 3 and out.shape[2] >= 5:
        pred = out[0]  # [num_boxes, 4 + C]
        boxes_cxcywh = pred[:, :4]
        scores_all = pred[:, 4:]
        if scores_all.shape[-1] == 1:
            scores = scores_all[:, 0]
            labels = torch.zeros_like(scores, dtype=torch.long)
        else:
            scores, labels = scores_all.max(dim=-1)
            
        # If scores look un-activated, apply sigmoid
        if scores.max() > 1.0 or scores.min() < 0.0:
            scores = scores.sigmoid()
            
        keep = scores > conf_thresh
        if keep.sum() == 0:
            return {"boxes": torch.zeros((0, 4)), "scores": torch.zeros((0,)), "labels": torch.zeros((0,), dtype=torch.long)}
            
        boxes_xyxy = box_cxcywh_to_xyxy(boxes_cxcywh[keep])
        scores_k = scores[keep]
        labels_k = labels[keep]
        
        # Apply Batched NMS
        from torchvision.ops import batched_nms
        nms_keep = batched_nms(boxes_xyxy, scores_k, labels_k, iou_thresh)
        boxes_xyxy = boxes_xyxy[nms_keep]
        scores_k = scores_k[nms_keep]
        labels_k = labels_k[nms_keep]
        
        # Scale from model resolution coordinate space to original image
        scale_x = orig_w / float(plan_resolution)
        scale_y = orig_h / float(plan_resolution)
        scaled_boxes = boxes_xyxy * torch.tensor([scale_x, scale_y, scale_x, scale_y], device=boxes_xyxy.device)
        
        return {
            "boxes": scaled_boxes.cpu().float(),
            "scores": scores_k.cpu().float(),
            "labels": labels_k.cpu().long(),
        }
        
    # Case 3: 2-tensor output (DETR-style: [boxes, logits])
    if len(raw_outputs) >= 2:
        return decode_rfdetr_outputs({"pred_logits": raw_outputs[1], "pred_boxes": raw_outputs[0]}, orig_size, conf_thresh)
        
    return {"boxes": torch.zeros((0, 4)), "scores": torch.zeros((0,)), "labels": torch.zeros((0,), dtype=torch.long)}

logger.info("Unified prediction decoders compiled.")


In [ ]:
# CELL 7 - Quantitative Accuracy Evaluation (COCO mAP)
logger.info("=" * 70)
logger.info("[CELL 7] Computing COCO mAP Metrics for Both Models...")
logger.info("=" * 70)

def evaluate_model_map(model_name: str, runner_type: str, dataset: COCOEvalDataset) -> Dict[str, Any]:
    logger.info(f"--> Evaluating {model_name} on {len(dataset)} samples...")
    metric = MeanAveragePrecision(iou_type="bbox", class_metrics=True)
    
    pbar = tqdm(total=len(dataset), desc=f"Eval {model_name}")
    for sample in dataset:
        orig_size = sample["orig_size"]
        gt_boxes = sample["boxes_xyxy_orig"]
        gt_labels = sample["labels"]
        
        # Run inference
        if runner_type == "rfdetr":
            if rfdetr_model is None:
                pbar.update(); continue
            inp = sample["norm_tensor"].unsqueeze(0).to(device)
            with torch.no_grad():
                with torch.amp.autocast(device_type="cuda", enabled=torch.cuda.is_available()):
                    out = rfdetr_model(inp)
            preds = decode_rfdetr_outputs(out, orig_size, conf_thresh=CONFIDENCE_THRESHOLD)
            
        elif runner_type == "trt":
            if not trt_runner.is_ready:
                pbar.update(); continue
            # Resize to TRT engine resolution
            raw_t = sample["raw_rgb_tensor"]
            if RESOLUTION_PLAN != RESOLUTION_RFDETR:
                inp = TF.resize(raw_t, [RESOLUTION_PLAN, RESOLUTION_PLAN]).unsqueeze(0).to(device)
            else:
                inp = raw_t.unsqueeze(0).to(device)
            with torch.no_grad():
                raw_outs = trt_runner.infer(inp)
            preds = decode_plan_outputs(raw_outs, orig_size, RESOLUTION_PLAN, conf_thresh=CONFIDENCE_THRESHOLD, iou_thresh=IOU_THRESHOLD)
            
        # Format for torchmetrics
        preds_dict = [{
            "boxes": preds["boxes"].float(),
            "scores": preds["scores"].float(),
            "labels": preds["labels"].long()
        }]
        targets_dict = [{
            "boxes": gt_boxes.float(),
            "labels": gt_labels.long()
        }]
        metric.update(preds_dict, targets_dict)
        pbar.update()
        
    pbar.close()
    computed = metric.compute()
    return computed

# 1. Evaluate New Model (RF-DETR)
results_rfdetr = {}
if rfdetr_model is not None:
    results_rfdetr = evaluate_model_map("New Model (RF-DETR)", "rfdetr", eval_dataset)
else:
    logger.warning("RF-DETR model not loaded; skipping its evaluation.")

# 2. Evaluate Old Model (model.plan)
results_plan = {}
if trt_runner.is_ready:
    results_plan = evaluate_model_map("Old Model (model.plan)", "trt", eval_dataset)
else:
    logger.warning("TensorRT model.plan engine not ready; skipping its evaluation.")


In [ ]:
# CELL 8 - GPU Speed & Latency Profiling (Warmup + 100 Iterations)
logger.info("=" * 70)
logger.info(f"[CELL 8] Benchmarking Inference Latency (Iterations={BENCHMARK_ITERS})...")
logger.info("=" * 70)

def benchmark_speed(model_name: str, run_fn, dummy_input) -> Dict[str, float]:
    logger.info(f"Benchmarking {model_name}...")
    # Warm-up passes
    for _ in range(10):
        run_fn(dummy_input)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        
    timings = []
    for _ in range(BENCHMARK_ITERS):
        t0 = time.perf_counter()
        run_fn(dummy_input)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        t1 = time.perf_counter()
        timings.append((t1 - t0) * 1000.0)  # in milliseconds
        
    timings = np.array(timings)
    mean_lat = float(np.mean(timings))
    median_lat = float(np.median(timings))
    p95_lat = float(np.percentile(timings, 95))
    fps = 1000.0 / mean_lat if mean_lat > 0 else 0.0
    
    logger.info(f"   - Mean Latency:   {mean_lat:.2f} ms")
    logger.info(f"   - Median Latency: {median_lat:.2f} ms")
    logger.info(f"   - P95 Latency:    {p95_lat:.2f} ms")
    logger.info(f"   - Throughput:     {fps:.1f} FPS")
    return {"mean_ms": mean_lat, "median_ms": median_lat, "p95_ms": p95_lat, "fps": fps}

speed_rfdetr = {}
if rfdetr_model is not None:
    dummy_rf = torch.randn(1, 3, RESOLUTION_RFDETR, RESOLUTION_RFDETR, device=device)
    def run_rf(inp):
        with torch.no_grad():
            with torch.amp.autocast(device_type="cuda", enabled=torch.cuda.is_available()):
                return rfdetr_model(inp)
    speed_rfdetr = benchmark_speed("RF-DETR", run_rf, dummy_rf)

speed_plan = {}
if trt_runner.is_ready:
    dummy_trt = torch.randn(1, 3, RESOLUTION_PLAN, RESOLUTION_PLAN, device=device)
    def run_trt(inp):
        with torch.no_grad():
            return trt_runner.infer(inp)
    speed_plan = benchmark_speed("model.plan (TRT)", run_trt, dummy_trt)


In [ ]:
# CELL 9 - Side-by-Side Performance Comparison Table
logger.info("=" * 70)
logger.info("[CELL 9] Compiling Comprehensive Metrics Comparison Table...")
logger.info("=" * 70)

def extract_metric(res_dict, key):
    v = res_dict.get(key, None)
    if v is None: return np.nan
    if isinstance(v, torch.Tensor): return float(v.item())
    return float(v)

metric_keys = [
    ("map", "mAP @ 50:95 (COCO)"),
    ("map_50", "mAP @ 50 (PASCAL VOC)"),
    ("map_75", "mAP @ 75 (Strict)"),
    ("map_small", "mAP Small Objects"),
    ("map_medium", "mAP Medium Objects"),
    ("map_large", "mAP Large Objects"),
    ("mar_100", "mAR (Average Recall)"),
]

rows = []
for key, label in metric_keys:
    v_plan = extract_metric(results_plan, key)
    v_rf = extract_metric(results_rfdetr, key)
    delta = (v_rf - v_plan) if not (np.isnan(v_plan) or np.isnan(v_rf)) else np.nan
    pct_delta = ((v_rf - v_plan) / max(v_plan, 1e-6) * 100) if not (np.isnan(v_plan) or np.isnan(v_rf)) else np.nan
    rows.append({
        "Metric": label,
        "Old Model (model.plan)": f"{v_plan:.4f}" if not np.isnan(v_plan) else "N/A",
        "New Model (RF-DETR)": f"{v_rf:.4f}" if not np.isnan(v_rf) else "N/A",
        "Absolute Delta": f"{delta:+.4f}" if not np.isnan(delta) else "N/A",
        "Relative Improvement": f"{pct_delta:+.2f}%" if not np.isnan(pct_delta) else "N/A",
    })

# Add Latency & Speed rows
lat_plan = speed_plan.get("mean_ms", np.nan)
lat_rf = speed_rfdetr.get("mean_ms", np.nan)
lat_delta = (lat_rf - lat_plan) if not (np.isnan(lat_plan) or np.isnan(lat_rf)) else np.nan
rows.append({
    "Metric": "Mean Latency (ms)",
    "Old Model (model.plan)": f"{lat_plan:.2f} ms" if not np.isnan(lat_plan) else "N/A",
    "New Model (RF-DETR)": f"{lat_rf:.2f} ms" if not np.isnan(lat_rf) else "N/A",
    "Absolute Delta": f"{lat_delta:+.2f} ms" if not np.isnan(lat_delta) else "N/A",
    "Relative Improvement": f"{(lat_plan - lat_rf)/max(lat_plan, 1e-6)*100:+.1f}% faster" if not (np.isnan(lat_plan) or np.isnan(lat_rf)) else "N/A",
})

fps_plan = speed_plan.get("fps", np.nan)
fps_rf = speed_rfdetr.get("fps", np.nan)
fps_delta = (fps_rf - fps_plan) if not (np.isnan(fps_plan) or np.isnan(fps_rf)) else np.nan
rows.append({
    "Metric": "Throughput (FPS)",
    "Old Model (model.plan)": f"{fps_plan:.1f} FPS" if not np.isnan(fps_plan) else "N/A",
    "New Model (RF-DETR)": f"{fps_rf:.1f} FPS" if not np.isnan(fps_rf) else "N/A",
    "Absolute Delta": f"{fps_delta:+.1f} FPS" if not np.isnan(fps_delta) else "N/A",
    "Relative Improvement": f"{(fps_rf - fps_plan)/max(fps_plan, 1e-6)*100:+.1f}%" if not (np.isnan(fps_plan) or np.isnan(fps_rf)) else "N/A",
})

df_comparison = pd.DataFrame(rows)
csv_summary_path = OUTPUT_DIR / "model_comparison_summary.csv"
df_comparison.to_csv(csv_summary_path, index=False)

logger.info("\n" + df_comparison.to_string(index=False))
logger.info(f"Saved complete comparison summary table to: {csv_summary_path}")


In [ ]:
# CELL 10 - 3-Panel Visual Comparison: Ground Truth vs Old Model vs New Model
logger.info("=" * 70)
logger.info(f"[CELL 10] Generating 3-Panel Visual Comparisons ({NUM_VISUAL_PREVIEWS} samples)...")
logger.info("=" * 70)

# Supervision palette & annotators
color_palette = sv.ColorPalette.from_hex([
    "#e6194B", "#3cb44b", "#ffe119", "#4363d8", "#f58231",
    "#911eb4", "#42d4f4", "#f032e6", "#bfef45", "#fabed4", "#469990"
])
box_annotator = sv.BoxAnnotator(color=color_palette, thickness=2)
label_annotator = sv.LabelAnnotator(color=color_palette, text_scale=0.5, text_thickness=1)

def annotate_image(img_bgr: np.ndarray, boxes_xyxy: np.ndarray, labels_idx: np.ndarray, scores: Optional[np.ndarray] = None, title: str = "") -> np.ndarray:
    annotated = img_bgr.copy()
    if len(boxes_xyxy) > 0:
        detections = sv.Detections(
            xyxy=boxes_xyxy,
            confidence=scores if scores is not None else np.ones(len(boxes_xyxy)),
            class_id=labels_idx
        )
        if scores is not None:
            text_labels = [f"{class_names[c] if c < len(class_names) else str(c)} {s:.2f}" for c, s in zip(labels_idx, scores)]
        else:
            text_labels = [f"{class_names[c] if c < len(class_names) else str(c)} [GT]" for c in labels_idx]
            
        annotated = box_annotator.annotate(scene=annotated, detections=detections)
        annotated = label_annotator.annotate(scene=annotated, detections=detections, labels=text_labels)
        
    # Draw title banner on top
    banner = np.zeros((36, annotated.shape[1], 3), dtype=np.uint8)
    cv2.putText(banner, title, (10, 24), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2, cv2.LINE_AA)
    return np.vstack([banner, annotated])

preview_count = 0
for i, sample in enumerate(eval_dataset):
    if preview_count >= NUM_VISUAL_PREVIEWS:
        break
        
    orig_h, orig_w = sample["orig_size"]
    img_path = os.path.join(EVAL_IMAGES_DIR, sample["file_name"])
    base_bgr = cv2.imread(img_path)
    if base_bgr is None: continue
    
    # 1. Ground Truth
    gt_boxes = sample["boxes_xyxy_orig"].numpy()
    gt_labels = sample["labels"].numpy()
    panel_gt = annotate_image(base_bgr, gt_boxes, gt_labels, scores=None, title=f"Ground Truth ({len(gt_boxes)} boxes)")
    
    # 2. Old Model (model.plan)
    if trt_runner.is_ready:
        raw_t = sample["raw_rgb_tensor"]
        inp = TF.resize(raw_t, [RESOLUTION_PLAN, RESOLUTION_PLAN]).unsqueeze(0).to(device) if RESOLUTION_PLAN != RESOLUTION_RFDETR else raw_t.unsqueeze(0).to(device)
        with torch.no_grad():
            raw_outs = trt_runner.infer(inp)
        plan_preds = decode_plan_outputs(raw_outs, (orig_h, orig_w), RESOLUTION_PLAN, conf_thresh=CONFIDENCE_THRESHOLD, iou_thresh=IOU_THRESHOLD)
        plan_boxes = plan_preds["boxes"].numpy()
        plan_scores = plan_preds["scores"].numpy()
        plan_labels = plan_preds["labels"].numpy()
        panel_plan = annotate_image(base_bgr, plan_boxes, plan_labels, plan_scores, title=f"Old Model: model.plan ({len(plan_boxes)} detected)")
    else:
        panel_plan = annotate_image(base_bgr, np.zeros((0, 4)), np.zeros(0, dtype=int), title="Old Model: model.plan (Not Loaded)")
        
    # 3. New Model (RF-DETR)
    if rfdetr_model is not None:
        inp = sample["norm_tensor"].unsqueeze(0).to(device)
        with torch.no_grad():
            with torch.amp.autocast(device_type="cuda", enabled=torch.cuda.is_available()):
                rf_outs = rfdetr_model(inp)
        rf_preds = decode_rfdetr_outputs(rf_outs, (orig_h, orig_w), conf_thresh=CONFIDENCE_THRESHOLD)
        rf_boxes = rf_preds["boxes"].numpy()
        rf_scores = rf_preds["scores"].numpy()
        rf_labels = rf_preds["labels"].numpy()
        panel_rf = annotate_image(base_bgr, rf_boxes, rf_labels, rf_scores, title=f"New Model: RF-DETR ({len(rf_boxes)} detected)")
    else:
        panel_rf = annotate_image(base_bgr, np.zeros((0, 4)), np.zeros(0, dtype=int), title="New Model: RF-DETR (Not Loaded)")
        
    # Match heights and concatenate horizontally
    target_h = 500
    def resize_h(p):
        aspect = p.shape[1] / p.shape[0]
        return cv2.resize(p, (int(target_h * aspect), target_h))
        
    combined = np.hstack([resize_h(panel_gt), resize_h(panel_plan), resize_h(panel_rf)])
    preview_path = PREVIEWS_DIR / f"comparison_sample_{preview_count + 1}.jpg"
    cv2.imwrite(str(preview_path), combined)
    logger.info(f"Preview #{preview_count + 1} saved -> {preview_path}")
    display(IPImage(filename=str(preview_path)))
    preview_count += 1

logger.info(f"Successfully exported {preview_count} side-by-side comparison images to {PREVIEWS_DIR}.")


In [ ]:
# CELL 11 - Graphical Performance Comparisons (mAP & FPS)
logger.info("=" * 70)
logger.info("[CELL 11] Plotting Performance Charts...")
logger.info("=" * 70)

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

models = ["Old Model (model.plan)", "New Model (RF-DETR)"]
map50_vals = [extract_metric(results_plan, "map_50"), extract_metric(results_rfdetr, "map_50")]
map_vals = [extract_metric(results_plan, "map"), extract_metric(results_rfdetr, "map")]
fps_vals = [speed_plan.get("fps", 0.0), speed_rfdetr.get("fps", 0.0)]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Chart 1: Accuracy (mAP)
x = np.arange(len(models))
width = 0.35
rects1 = ax1.bar(x - width/2, [v if not np.isnan(v) else 0 for v in map50_vals], width, label="mAP @ 0.50", color="#4363d8")
rects2 = ax1.bar(x + width/2, [v if not np.isnan(v) else 0 for v in map_vals], width, label="mAP @ 0.50:0.95", color="#3cb44b")
ax1.set_ylabel("Score")
ax1.set_title("Detection Accuracy Comparison")
ax1.set_xticks(x)
ax1.set_xticklabels(models)
ax1.legend()
ax1.set_ylim(0, 1.05)
for r in rects1 + rects2:
    h = r.get_height()
    if h > 0:
        ax1.annotate(f"{h:.3f}", xy=(r.get_x() + r.get_width() / 2, h), xytext=(0, 3), textcoords="offset points", ha="center", va="bottom", fontsize=9)

# Chart 2: Throughput (FPS)
rects3 = ax2.bar(models, fps_vals, color=["#f58231", "#911eb4"], width=0.45)
ax2.set_ylabel("Frames Per Second (FPS)")
ax2.set_title("Inference Throughput Benchmark (Higher is Better)")
for r in rects3:
    h = r.get_height()
    if h > 0:
        ax2.annotate(f"{h:.1f} FPS", xy=(r.get_x() + r.get_width() / 2, h), xytext=(0, 3), textcoords="offset points", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
chart_save_path = OUTPUT_DIR / "benchmark_charts.png"
plt.savefig(str(chart_save_path), dpi=150, bbox_inches="tight")
plt.close(fig)
logger.info(f"Performance comparison charts saved to: {chart_save_path}")
display(IPImage(filename=str(chart_save_path)))


## Final Comparison Summary & Next Steps

- **Summary Table CSV**: Saved to `evaluation_comparison/model_comparison_summary.csv`.
- **Benchmark Charts**: Saved to `evaluation_comparison/benchmark_charts.png`.
- **Visual Previews**: Saved to `evaluation_comparison/previews/` for comprehensive qualitative inspection.

### Recommended Decision Matrix:
1. **Accuracy Superiority**: Check if RF-DETR delivers higher mAP@50 and mAP@50:95, particularly on challenging objects (small/partially occluded).
2. **Throughput / Latency**: Review if `model.plan` (TensorRT optimized) maintains a speed advantage, and consider exporting the fine-tuned RF-DETR model to TensorRT/ONNX if edge deployment requires matching throughput.
3. **Error Disagreements**: Review `comparison_sample_*.jpg` in the `evaluation_comparison/previews/` folder to check which model generalizes better to false positive distractors.